# Visualizing MammAlps predictions and ground truth

This notebook renders bounding boxes and attributes (species, action, activity,
demographics) on top of a MammAlps video, either from the **ground-truth
annotations** shipped with the
[amathislab/Prompting-MammAlps](https://huggingface.co/datasets/amathislab/Prompting-MammAlps)
dataset, or from a **prediction JSON**.

## 1. Download a sample from the dataset

The full dataset is ~79GB (videos + annotations), so for this tutorial we only
download one camera's worth of videos/annotations plus the small metadata
files, using `allow_patterns`. Adjust `SITE_CAMERA` or drop `allow_patterns`
entirely to fetch more/all of the data.

In [ ]:
import os
import json
from pathlib import Path

import ffmpeg
import numpy as np
import supervision as sv
from huggingface_hub import snapshot_download
from tqdm.auto import tqdm

REPO_ID = "amathislab/Prompting-MammAlps"
DATA_ROOT = Path("./data")
SITE_CAMERA = "S2/C2"  # one camera's worth of data, enough for this demo

snapshot_download(
    repo_id=REPO_ID,
    repo_type="dataset",
    local_dir=DATA_ROOT,
    allow_patterns=[
        "metadata/label_mapping.json",
        f"videos/test/{SITE_CAMERA}/*",
        f"annotations/test/{SITE_CAMERA}/*",
    ],
)

with open(DATA_ROOT / "metadata" / "label_mapping.json", "r") as f:
    label_mapping = json.load(f)

Fetching 73 files: 100%|██████████| 73/73 [00:12<00:00,  5.89it/s]


## 2. Helper functions

`process_video` reads a video and its per-frame detections JSON
(`{"frames": [{"detections": [{"bbox", "track_id", "conf", "attributes": {...}}, ...]}, ...]}`)
and writes an annotated copy. `color_by` controls whether boxes are colored by
track identity, action, or activity.

In [ ]:
def from_detection_to_sv(frame_detection_results, color_by) -> sv.Detections:
    if frame_detection_results["detections"]:
        detections_list = frame_detection_results.get("detections", [])
        xyxy_coord = np.array([d["bbox"] for d in detections_list], dtype=int) # in px
        tracker_id = np.array([int(d["track_id"]) for d in detections_list])

        extra_data = {}
        for attribute_key in ["Species", "Deer_age", "Deer_adult_sex", "Activity", "Action", "Action2"]:
            values = [d["attributes"].get(attribute_key) if "attributes" in d.keys() else None for d in detections_list]
            extra_data[attribute_key] = np.array(values)

        if color_by == "action":
            class_id = np.array([int(label_mapping["actions"][d["attributes"]["Action"]]) for d in detections_list])
        elif color_by == "activity":
            class_id = np.array([int(label_mapping["activities"][d["attributes"]["Activity"]]) for d in detections_list])
        elif color_by == "track":
            class_id = np.array([int(d["track_id"]) for d in detections_list])

        detections = sv.Detections(
            xyxy=xyxy_coord,
            class_id=class_id,
            tracker_id=tracker_id,
            data=extra_data,
        )
    else:
        detections = sv.Detections.empty()

    return detections


def process_video(
    source_video_path: str,
    detections_path: str,
    target_video_path: str,
    color_by: str,
) -> None:

    # Supervision Annotator objects
    color_lookup = sv.ColorLookup("class")
    box_annotator = sv.BoxAnnotator(color_lookup=color_lookup, thickness=2)
    box_fill_annotator = sv.ColorAnnotator(color_lookup=color_lookup, opacity=0.2)
    label_annotator = sv.LabelAnnotator(text_scale=1, color_lookup=color_lookup, text_thickness=2, smart_position=True)
    
    # Generator for the source video
    frame_generator = sv.get_video_frames_generator(source_path=source_video_path, stride=1)
    video_info = sv.VideoInfo.from_video_path(video_path=source_video_path)

    with open(detections_path, "r") as f:
        detection_results = json.load(f)["frames"]

    frame_idx = 0
    with sv.VideoSink(
        target_path=target_video_path, video_info=video_info, codec="mp4v"
    ) as sink:
        for frame in tqdm(frame_generator):
            # Get annotations or predictions from JSON file in supervision object
            annotated_frame = frame.copy()
            detections = from_detection_to_sv(detection_results[frame_idx], color_by=color_by)

            # Prepare labels
            labels = []
            if detections.tracker_id is not None:
                for i, _ in enumerate(detections.tracker_id):
                    # Annotating labels
                    label_parts = []

                    # Add any extra fields dynamically
                    for key, values in detections.data.items():
                        if values is not None and len(values) > i:
                            value = values[i]
                            if isinstance(value, dict):
                                dict_string = "\n".join(
                                    [
                                        f"{k}:{v}"
                                        for k, v in value.items()
                                        if (v is not None and v != "none")
                                    ]
                                )
                                label_parts.append(dict_string)
                            elif value is not None and value != "none":
                                label_parts.append(f"{key}: {value}")

                    labels.append("\n".join(label_parts))

            # Annotating detection boxes
            annotated_frame = box_annotator.annotate(scene=annotated_frame, detections=detections)
            annotated_frame = box_fill_annotator.annotate(scene=annotated_frame, detections=detections)
            annotated_frame = label_annotator.annotate(scene=annotated_frame, detections=detections, labels=labels)

            sink.write_frame(frame=annotated_frame)

            frame_idx += 1

def reencode_video_w_audio(target_video_path: str, orig_video_path: str, codec="libx264"):
    coded_video_path = str(Path(target_video_path).parent / (Path(target_video_path).stem + "_" + codec + ".mp4"))
    (
        ffmpeg.input(target_video_path).video
        .output(
            ffmpeg.input(orig_video_path).audio,
            coded_video_path,
            vcodec="libx264",
            pix_fmt="yuv420p",
            video_bitrate="12048k",
            acodec="copy",
            **{"profile:v": "high"},
        )
        .run(quiet=True)
    )
    os.remove(target_video_path)
    os.rename(coded_video_path, target_video_path)

## 3. Visualize the ground-truth annotations

Pick any video id present under the downloaded `videos/test/S2/C3/` folder —
its annotation JSON lives at the matching path under `annotations/`.

In [21]:
video_id = "S2_C2_F567_V0082"

video_path = DATA_ROOT / "videos" / "test" / SITE_CAMERA / f"{video_id}.mp4"
annot_path = DATA_ROOT / "annotations" / "test" / SITE_CAMERA / f"{video_id}.json"
output_path = f"./{video_id}_gt.mp4"

process_video(str(video_path), str(annot_path), output_path, color_by="track")
reencode_video_w_audio(output_path, str(video_path))

705it [00:12, 58.62it/s]
